In [1]:
# 最小评估器示例：数值检验 Cauchy-Schwarz 不等式 |<x,y>| <= ||x||*||y||
# 评估器本身就是几行确定性代码，其正确性可以直接审查，这是它可信的来源
import numpy as np

rng = np.random.default_rng(2026)          # 固定种子：评估结果可复现

def evaluator_cauchy_schwarz(n_trials=1000, dim=50):
    """在随机向量上数值检验 Cauchy-Schwarz，返回观测到的最大比值。"""
    worst = 0.0
    for _ in range(n_trials):
        x = rng.normal(size=dim)
        y = rng.normal(size=dim)
        ratio = abs(x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))
        worst = max(worst, ratio)
    return worst                            # 理论上不超过 1

print(evaluator_cauchy_schwarz())           # 实测约 0.43，始终 <= 1

0.42873484039764725


In [2]:
# ECE 小示例：分组经验正确率与平均置信度的加权绝对偏差
import numpy as np

rng = np.random.default_rng(7)

def ece(conf, correct, n_bins=10):
    """conf: 模型自评置信度（[0,1]）；correct: 0/1 是否答对。"""
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.sum() == 0:
            continue
        e += m.sum() / len(conf) * abs(correct[m].mean() - conf[m].mean())
    return e

n = 5000
conf = rng.uniform(0.5, 1.0, n)
for name, p_true in [("过度自信（真实正确率=置信度平方）", conf ** 2),
                     ("接近校准（真实正确率=置信度）", conf)]:
    correct = rng.binomial(1, p_true)
    print(f"{name}: ECE = {ece(conf, correct):.4f}")

过度自信（真实正确率=置信度平方）: ECE = 0.1796
接近校准（真实正确率=置信度）: ECE = 0.0054


In [3]:
# 模型塌缩的最小模拟：迭代"拟合高斯 -> 从拟合分布采样 -> 用样本重新拟合"
# 真实分布是标准正态；n=20 让估计误差的逐代累积显形；300 次重复取均值去噪
import numpy as np

rng = np.random.default_rng(1)
n, reps, gens = 20, 300, 10

def simulate(mix_frac=0.0):
    x = rng.normal(size=(reps, n))                # 第 0 代：真实数据 N(0,1)
    sds = [x.std(axis=1).mean()]
    for _ in range(gens):
        mu = x.mean(axis=1, keepdims=True)        # "训练"= 逐条拟合高斯
        sd = x.std(axis=1, keepdims=True)
        n_real = int(round(n * mix_frac))
        synth = rng.normal(mu, sd, size=(reps, n - n_real))
        real = rng.normal(size=(reps, n_real))    # 真实数据仍来自 N(0,1)
        x = np.concatenate([synth, real], axis=1)
        sds.append(x.std(axis=1).mean())
    return sds

pure  = simulate(0.0)
mixed = simulate(0.2)
for g in range(gens + 1):
    print(f"gen {g:2d}  纯合成 sd≈{pure[g]:.3f}   混入20%真实 sd≈{mixed[g]:.3f}")

gen  0  纯合成 sd≈0.963   混入20%真实 sd≈0.965
gen  1  纯合成 sd≈0.922   混入20%真实 sd≈0.945
gen  2  纯合成 sd≈0.879   混入20%真实 sd≈0.933
gen  3  纯合成 sd≈0.831   混入20%真实 sd≈0.913
gen  4  纯合成 sd≈0.785   混入20%真实 sd≈0.904
gen  5  纯合成 sd≈0.754   混入20%真实 sd≈0.909
gen  6  纯合成 sd≈0.733   混入20%真实 sd≈0.912
gen  7  纯合成 sd≈0.709   混入20%真实 sd≈0.922
gen  8  纯合成 sd≈0.678   混入20%真实 sd≈0.906
gen  9  纯合成 sd≈0.657   混入20%真实 sd≈0.919
gen 10  纯合成 sd≈0.634   混入20%真实 sd≈0.909
